<a href="https://colab.research.google.com/github/esalinasbio/taller-modelado-biomolecular/blob/master/notebooks/01_Dinamica_Molecular_fix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ejercicio 1 - Dinámica Molecular de un complejo proteína–RNA

**Taller de modelado biomolecular**

---

### El sistema: U1A unida a su propio 3'UTR (PDB `1AUD`, RMN)

U1A es una proteína del snRNP U1, el complejo que inicia el *splicing* reconociendo el sitio de corte 5' de los intrones. Se une a RNA mediante un dominio **RRM** (*RNA Recognition Motif*), el módulo de unión a RNA más común en
eucariotas. Lo interesante es que U1A también reconoce una segunda secuencia: la de **su propio mRNA**, en el 3'UTR. Cuando se une inhibe la poliadenilación de su propio mensajero — un circuito de retroalimentación negativa construido enteramente con reconocimiento molecular.
`1AUD` es la estructura de ese segundo complejo: el RRM unido a 30 nucleótidos
de su propio 3'UTR, que forman una horquilla con un *loop* flexible sobre el que se acuesta la hoja-β de la proteína.

---

### Qué vamos a hacer

| Paso | Qué hacemos | Qué concepto de DM ilustra |
|---|---|---|
| 1 | Elegir modelo del ensamble | La RMN da muchos modelos: ¿cuál es *la* estructura? |
| 2 | Protonar | Los hidrógenos no vienen del experimento |
| 3 | Asignar campo de fuerza | Qué es realmente un campo de fuerza |
| 4 | Solvatar y añadir iones | Periodicidad, imagen mínima, forma de caja, contraiones |
| 5 | Minimizar | Minimización **no es** dinámica |
| 6 | Equilibrar NVT | Maxwell–Boltzmann, termostato, restricciones |
| 7 | Equilibrar NPT | Barostato, densidad, ¿qué significa "equilibrado"? |
| 8 | Producción | Paso de integración, restricciones de enlace, muestreo |
| 9 | Centrar y reimaginar | Qué significa "reimaginar" una trayectoria periódica |

---

> ### Cómo usar este cuaderno
>
> **No se trata de correr las celdas hasta el final.** En cada paso van a *ver* el sistema y a leer un número con significado físico. Antes de varias celdas hay una
>
> **Preguntas**: deténganse diez segundos, piensen en su respuesta, y después
> ejecuten.
>
> Los parámetros están expuestos como campos editables, no escondidos en el código. Este cuaderno está pensado para que se lo lleven y lo puedan usar con su propio sistema (Aunque es muy básico).

---
## Paso 0 - Entorno

* **`Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → `T4 GPU`**.

* Presiona `Ctrl + S` o usa el menú `Archivo` (`File`) para guardar una copia del notebook a tu Google Drive

Ahora ejecuten la instalación **inmediatamente**. Tarda ~90 s y corre en segundo
plano mientras platicamos del sistema.

In [ ]:
#@title Instalar dependencias
import time; _t0 = time.time()
import subprocess
import sys
import os
print("installing conda...")
os.system("wget -qnc https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh")
os.system("bash Miniforge3-Linux-x86_64.sh -bfp /usr/local")
os.system("mamba config --set auto_update_conda false")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "openmm[cuda12]",
        "pdbfixer",
        "mdtraj",
        "py3Dmol"
    ],
    check=True
)
subprocess.run("conda install -c conda-forge ambertools", shell=True)
print(f"Instalación terminada en {time.time()-_t0:.0f} s")

import openmm as mm
from openmm import unit
print("OpenMM version:", mm.version.version, "\n")
for i in range(mm.Platform.getNumPlatforms()):
    p = mm.Platform.getPlatform(i)
    print(f"  [{i}] {p.getName():10s}  velocidad relativa = {p.getSpeed()}")
nombres = [mm.Platform.getPlatform(i).getName() for i in range(mm.Platform.getNumPlatforms())]
print("\nCUDA disponible — vamos bien." if "CUDA" in nombres else
      "\n*** NO hay CUDA. Cambien el entorno a GPU T4 y ejecuten otra vez. ***")

import py3Dmol, numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

C_PROT, C_RNA, C_AGUA, C_ION, GRIS = "#2E8B8B", "#E8813A", "#9fc7d8", "#c8a02c", "#64748b"

def _aplicar(v, estilos, viewer=None):
    for k, (sel, est) in enumerate(estilos):
        kw = {} if viewer is None else {"viewer": viewer}
        (v.setStyle if k == 0 else v.addStyle)(sel, est, **kw)

def ver(archivo, estilos, ancho=780, alto=430, girar=False, celda=False):
    v = py3Dmol.view(width=ancho, height=alto)
    v.addModel(open(archivo).read(), "pdb")
    _aplicar(v, estilos)
    if celda: v.addUnitCell({"box": {"color": "#94a3b8"}})
    v.zoomTo()
    if girar: v.spin(True)
    v.show(); return v

def ver_lado_a_lado(a_izq, e_izq, a_der, e_der, ancho=880, alto=400):
    v = py3Dmol.view(width=ancho, height=alto, viewergrid=(1, 2))
    v.addModel(open(a_izq).read(), "pdb", viewer=(0, 0)); _aplicar(v, e_izq, viewer=(0, 0))
    v.addModel(open(a_der).read(), "pdb", viewer=(0, 1)); _aplicar(v, e_der, viewer=(0, 1))
    v.zoomTo(); v.show(); return v

def ver_animacion(archivo_multi, estilos, ancho=780, alto=440, intervalo=150):
    v = py3Dmol.view(width=ancho, height=alto)
    v.addModelsAsFrames(open(archivo_multi).read(), "pdb")
    _aplicar(v, estilos)
    v.zoomTo(); v.animate({"loop": "forward", "interval": intervalo}); v.show(); return v

COMPLEJO = [({"chain": "A"}, {"cartoon": {"color": C_PROT}}),
            ({"chain": "B"}, {"cartoon": {"color": C_RNA}}),
            ({"chain": "B"}, {"stick": {"radius": 0.15, "color": C_RNA}})]



installing conda...
Instalación terminada en 90 s
OpenMM version: 8.6.0.dev-c6173db 

  [0] Reference   velocidad relativa = 1.0
  [1] CPU         velocidad relativa = 10.0
  [2] CUDA        velocidad relativa = 100.0
  [3] OpenCL      velocidad relativa = 50.0

CUDA disponible — vamos bien.


---
## Paso 1 - El ensamble de RMN

Un cristal da **una** densidad electrónica, y de ahí sale un modelo. La RMN da un conjunto de modelos, todos igualmente compatibles con los datos. Lo que se deposita es un *ensamble*.

Entonces: ¿cuál de todos es "la estructura"?

In [ ]:
#@title Descargar la estructura
PDB_ID = "1AUD"   #@param {type:"string"}

!wget -q https://files.rcsb.org/download/{PDB_ID}.pdb -O entrada.pdb

from openmm.app import (PDBFile, Modeller, ForceField, PME, HBonds, AllBonds,
                        HAngles, Simulation, DCDReporter, StateDataReporter, element)
import numpy as np

crudo = PDBFile("entrada.pdb")
n_modelos = crudo.getNumFrames()


print(f"Modelos depositados : {n_modelos}")
print(f"Átomos por modelo   : {crudo.topology.getNumAtoms()}\n")
print(f"{'cadena':<10}{'tipo':<10}{'residuos':>10}{'rango resSeq':>20}")
print("-"*50)
for ch in crudo.topology.chains():
    res = [r for r in ch.residues() if r.name.strip() not in ("HOH", "WAT")]
    if not res: continue
    es_rna = res[0].name.strip() in ("A", "U", "G", "C")
    ids = [int(r.id) for r in res]
    print(f"{ch.id:<9}{'RNA' if es_rna else 'proteína':<12}"
          f"{len(res):>10}{f'{min(ids)}-{max(ids)}':>16}")
if PDB_ID == '1AUD':
  print('''
  Un detalle de numeración: el RNA tiene 30 nucleótidos pero los números van del
  19 al 50. No falta nada, la numeración sigue la del 3'-UTR natural, no la del
  construto sintetizado.
  ''')

ModuleNotFoundError: No module named 'openmm'

In [ ]:
#@title VER - el ensamble animado, modelo por modelo
ver_animacion("entrada.pdb",
              [({"chain": "A"}, {"cartoon": {"color": C_PROT, "opacity": 0.9}}),
               ({"chain": "B"}, {"cartoon": {"color": C_RNA, "opacity": 0.9}})],
              intervalo=200)
print("Cada frame es un modelo distinto. TODOS ajustan los datos experimentales igual de bien.")
print("Fíjense dónde se mueve más: ésa es la parte con mayor movilidad del complejo.")

NameError: name 'ver_animacion' is not defined

In [ ]:
#@title VER - todos los modelos superpuestos a la vez
v = py3Dmol.view(width=780, height=450)
v.addModels(open("entrada.pdb").read(), "pdb")
for k in range(n_modelos):
    v.setStyle({"model": k}, {"cartoon": {"colorscheme": "chain", "opacity": 1}})
v.zoomTo()
v.show()

P = np.array([crudo.getPositions(asNumpy=True, frame=k).value_in_unit(unit.nanometer)
              for k in range(n_modelos)])
disp = np.linalg.norm(P - P.mean(axis=0), axis=2).mean(axis=0)

if n_modelos > 1:
  print(f"\nDispersión del ensamble (desviación media respecto al promedio):")
  print(f"   global : {10*disp.mean():.2f} Å")
  print(f"   máxima : {10*disp.max():.2f} Å")

In [ ]:
#@title Seleccionar un modelo del ensamble
MODELO_NMR = 2   #@param {type:"integer"}

#@markdown Proteína (dominio RRM) en azul-verde, RNA (30 nt) en naranja.

if n_modelos == 1:
  MODELO_NMR = 1

idx = max(0, min(MODELO_NMR - 1, n_modelos - 1))
mod = Modeller(crudo.topology, crudo.getPositions(frame=idx))
mod.deleteWater()
with open("01_modelo_elegido.pdb", "w") as f:
    PDBFile.writeFile(mod.topology, mod.positions, f)
print(f"Trabajando con el modelo {idx+1} de {n_modelos}. Átomos: {mod.topology.getNumAtoms()}")

COMPLEJO = [({"chain": "B"}, {"cartoon": {"color": C_PROT}}),
            ({"chain": "A"}, {"cartoon": {"color": C_RNA, 'opacity':0.75}}),
            ({"chain": "A"}, {"stick": {"radius": 0.15, "color": C_RNA}})]

ver("01_modelo_elegido.pdb", COMPLEJO)
print("")

In [ ]:
#@title VER - Complejor RNA-Proteína (superficie)
#@markdown Superficies independientes por cadena para visualizar el acoplamiento.
view = py3Dmol.view(width=800, height=600)
view.addModel(open('01_modelo_elegido.pdb', 'r').read(), 'pdb')
view.setStyle({'chain': 'A'}, {'cartoon': {'color': C_RNA, "opacity": 0.3}})
view.setStyle({'chain': 'B'}, {'cartoon': {'color': C_PROT, "opacity": 0.3}})
view.addSurface(
    py3Dmol.VDW,
    {'opacity': 1, 'color': C_RNA},
    {'chain': 'A'}
)
view.addSurface(
    py3Dmol.VDW,
    {'opacity': 1, 'color': C_PROT},
    {'chain': 'B'}
)
view.zoomTo()
view.show()

In [ ]:
# @title VER -  Complejo Proteina-RNA, interfaz
view = py3Dmol.view(width=900, height=700)
view.addModel(open('01_modelo_elegido.pdb', 'r').read(), 'pdb')

view.setStyle({'chain': 'A'}, {'cartoon': {'color': C_RNA, 'opacity': 0.85}})
view.setStyle({'chain': 'B'}, {'cartoon': {'color': C_PROT, 'opacity': 0.85}})

interface_RNA = {'and': [{'chain': 'A'}, {'within': {'distance': 5.0, 'sel': {'chain': 'B'}}}]}
interface_PROT = {'and': [{'chain': 'B'}, {'within': {'distance': 5.0, 'sel': {'chain': 'A'}}}]}

view.addStyle(interface_RNA, {'stick': {'colorscheme': 'cyanCarbon', 'radius': 0.20}})
view.addStyle(interface_PROT, {'stick': {'colorscheme': 'yellowCarbon', 'radius': 0.18}})

view.addStyle({'and': [{'chain': 'A'}, {'within': {'distance': 5.0, 'sel': {'chain': 'B'}}}, {'not': {'atom': 'P'}}]}, {'stick': {'colorscheme': 'cyanCarbon', 'radius': 0.23}})

view.zoomTo()
view.show()

---
## Paso 2 - Protonar

Las estructuras del PDB normalmente estan incompletas: sin hidrógenos, con cadenas laterales truncadas, residuos faltantes, etc. Reconstruir eso es una posible fuente de error.

1AUD no tiene cadenas laterales o regiones faltantes, pero si le hacen falta los hidrógenos.

> ### Pregunta
> Algunas estructuras pueden contener hidrógenos, entonces
>
> **1. ¿Deberíamos de removerlos? ¿Por qué no?**
>
> **2.¿Cómo sabemos dónde asignar los hidrógenos?**

In [ ]:
#@title Comprobar integridad y añadir protones
PH                  = 7.0             #@param {type:"number"}
CAMPO_FUERZA_SOLUTO = "amber14-all.xml"   #@param ["amber14-all.xml", "amber19-all.xml", "amber99sbildn.xml", "charmm36.xml", "charmm36_2024.xml"]
MODELO_AGUA         = "tip3p"             #@param ["tip3p", "opc3", "spce", "opc", "tip4pew"]

#@markdown - `PH`: pH al que se asignarán los estados de protonación
#@markdown - `CAMPO_FUERZA_SOLUTO`: Campo de fuerza a utilizar con los solutos (Proteína y RNA)
#@markdown - `MODELO_AGUA`: Modelo de agua a utilizar (los parámetros de los iones tambien están aquí.)

_GEOMETRIA = {"tip3p": "tip3p", "tip3pfb": "tip3p", "opc3": "tip3p", "spce": "spce",
              "tip4pew": "tip4pew", "tip4pfb": "tip4pew", "opc": "tip4pew",
              "tip4p2005": "tip4pew", "tip5p": "tip5p"}

_AMBER_MOD = ["tip3p", "tip3pfb", "opc3", "spce", "tip4pew", "tip4pfb", "opc"]
_AGUAS = {
    "amber14-all.xml":   {w: f"amber14/{w}.xml" for w in _AMBER_MOD},
    "amber19-all.xml":   {w: f"amber19/{w}.xml" for w in _AMBER_MOD},
    "amber99sbildn.xml": {w: f"{w}.xml" for w in _AMBER_MOD + ["tip5p"]},
    "charmm36.xml":      {w: f"charmm36/{w}.xml" for w in
                          ["spce", "tip4pew", "tip5p", "tip4p2005"]} | {"tip3p": "charmm36/water.xml"},
    "charmm36_2024.xml": {w: f"charmm36_2024/{w}.xml" for w in
                          ["spce", "tip4pew", "tip5p", "tip4p2005"]} | {"tip3p": "charmm36_2024/water.xml"},
}

if MODELO_AGUA not in _AGUAS[CAMPO_FUERZA_SOLUTO]:
    raise ValueError(
        f"'{MODELO_AGUA}' no existe para {CAMPO_FUERZA_SOLUTO}.\n"
        f"Disponibles: {sorted(_AGUAS[CAMPO_FUERZA_SOLUTO])}")

XML_AGUA  = _AGUAS[CAMPO_FUERZA_SOLUTO][MODELO_AGUA]
GEOM_AGUA = _GEOMETRIA[MODELO_AGUA]
print(f"Campo de fuerza: {CAMPO_FUERZA_SOLUTO}")
print(f"Modelo de agua: {XML_AGUA}")

from pdbfixer import PDBFixer
n_H_orig = sum(1 for a in mod.topology.atoms() if a.element == element.hydrogen)

fixer = PDBFixer(filename="01_modelo_elegido.pdb")

fixer.findMissingResidues()
cad = list(fixer.topology.chains())
internos = {k: v for k, v in fixer.missingResidues.items()
            if k[1] != 0 and k[1] != len(list(cad[k[0]].residues()))}
n_extremos = len(fixer.missingResidues) - len(internos)
fixer.missingResidues = internos

fixer.findNonstandardResidues(); fixer.replaceNonstandardResidues()
fixer.removeHeterogens(keepWater=False)
fixer.findMissingAtoms()
n_pesados = sum(len(v) for v in fixer.missingAtoms.values())

print("="*50)
print(f"Huecos internos a rellenar    : {len(fixer.missingResidues)}")
print(f"Extremos desordenados omitidos: {n_extremos}")
print(f"Átomos pesados faltantes      : {n_pesados}")
print(f"Residuos no estándar          : {fixer.nonstandardResidues}")
print("="*50)

fixer.addMissingAtoms()

sin_h = Modeller(fixer.topology, fixer.positions)
sin_h.delete([a for a in sin_h.topology.atoms() if a.element == element.hydrogen])
n_sin_H = sin_h.topology.getNumAtoms()
with open("02a_sin_hidrogenos.pdb", "w") as f:
    PDBFile.writeFile(sin_h.topology, sin_h.positions, f)

forcefield = ForceField(CAMPO_FUERZA_SOLUTO, XML_AGUA)
mod_h = Modeller(sin_h.topology, sin_h.positions)
mod_h.addHydrogens(forcefield, pH=PH)
with open("02b_protonado.pdb", "w") as f:
    PDBFile.writeFile(mod_h.topology, mod_h.positions, f)

n_H = sum(1 for a in mod_h.topology.atoms() if a.element == element.hydrogen)
print(f"\nHidrógenos en el archivo de RMN : {n_H_orig}")
print(f"Átomos pesados                  : {n_sin_H}")
print(f"Hidrógenos puestos a pH {PH}     : {n_H}")
print(f"Total                           : {mod_h.topology.getNumAtoms()}")
print(f"\nLos hidrógenos son el {100*n_H/mod_h.topology.getNumAtoms():.1f}% del número de átomos.")

In [ ]:
#@title VER · antes y después de protonar
v = py3Dmol.view(width=880, height=400, viewergrid=(1, 2))

v.addModel(open("02a_sin_hidrogenos.pdb").read(), "pdb", {"keepH": True}, viewer=(0, 0))
v.setStyle({}, {"stick": {"radius": 0.13, "color": GRIS}}, viewer=(0, 0))

v.addModel(open("02b_protonado.pdb").read(), "pdb", {"keepH": True}, viewer=(0, 1))
v.setStyle({}, {"stick": {"radius": 0.13, "color": GRIS}}, viewer=(0, 1))
v.addStyle({"elem": "H"}, {"sphere": {"radius": 0.25, "color": "#ef4444"}}, viewer=(0, 1))

v.zoomTo()
v.show()

print("Izquierda: sólo átomos pesados.   Derecha: con hidrógenos (rojo).")

---
## Paso 3 - El campo de fuerza

Un **campo de fuerza** es una expresión analítica de la energía potencial en función de las coordenadas. No es una aproximación a la mecánica cuántica, es una función empírica, parametrizada para reproducir observables concretos.

$$U = \underbrace{\sum_{\text{enlaces}} k_b (r-r_0)^2 + \sum_{\text{ángulos}} k_\theta (\theta-\theta_0)^2 + \sum_{\text{diedros}} \frac{V_n}{2}[1+\cos(n\phi-\gamma)]}_{\text{términos enlazados}} + \underbrace{\sum_{i<j} 4\epsilon_{ij}\left[\left(\frac{\sigma_{ij}}{r_{ij}}\right)^{12} - \left(\frac{\sigma_{ij}}{r_{ij}}\right)^{6}\right] + \sum_{i<j} \frac{q_i q_j}{4\pi\epsilon_0 r_{ij}}}_{\text{términos no enlazados}}$$

Todo lo que sucede durante la simulación se obtiene de derivar **esta** función. Nada más.

`amber14-all.xml` incluye **ff14SB** para la proteína y **OL3 (χOL3)** para el RNA.

> ### Pregunta
> El RNA tiene 30 nucleótidos. La proteína es básica.
> **¿Cuál es la carga total del complejo, y de qué signo?**

In [ ]:
#@title Asignar campo de fuerza y calcular la carga
mod2 = Modeller(mod_h.topology, mod_h.positions)
sistema_vacio = forcefield.createSystem(mod2.topology)
nbf_ = [f for f in sistema_vacio.getForces() if isinstance(f, mm.NonbondedForce)][0]

q_total = sum(nbf_.getParticleParameters(i)[0].value_in_unit(unit.elementary_charge)
              for i in range(sistema_vacio.getNumParticles()))
print("="*50)
for ch in mod2.topology.chains():
    q = sum(nbf_.getParticleParameters(a.index)[0].value_in_unit(unit.elementary_charge)
            for r in ch.residues() for a in r.atoms())
    res = list(ch.residues())
    tipo = "RNA" if res[0].name.strip() in ("A","U","G","C") else "proteína"
    print(f"  Carga de la cadena {ch.id} ({tipo:8s}) : {q:+7.2f} e")
print("-"*50)
print(f"  CARGA TOTAL DEL COMPLEJO        : {q_total:+7.2f} e")
print("="*50)

El esqueleto del RNA aporta una carga negativa por fosfato. Ésta es la razón física de por qué el RNA es sensible a la concentración iónica. La solvatación y ddistribución de cargas es una parte importante del modelo.

In [ ]:
#@title VER - Dónde están las cargas
#@markdown Rojo = fosfatos del RNA (negativos).  Azul = Arg y Lys (positivos).

#@markdown El reconocimiento proteína-RNA es, en buena medida, electrostático

v = py3Dmol.view(width=780, height=440)
v.addModel(open("02b_protonado.pdb").read(), "pdb")
v.setStyle({}, {"cartoon": {"color": "#e2e8f0"}})
v.addStyle({"atom": ["P", "OP1", "OP2"]}, {"sphere": {"radius": 0.5, "color": "#dc2626"}})
v.addStyle({"resn": ["ARG", "LYS"]}, {"stick": {"radius": 0.20, "color": "#2563eb"}})
v.zoomTo(); v.show()


> ### Nota sobre los parámetros (léanla, no la salten)
>
> **OL3 es el estándar en AMBER para RNA, pero no es un problema resuelto.** Puede tener dificultades para mantener estructuras terciarias plegadas, y las interfaces proteína–RNA no están tan validadas como las proteína–proteína.
> Combinaciones más recientes (por ejemplo OPC + OL3) mejoran algunas cosas y empeoran otras. Tómenlo como un tema **abierto**, no como una receta.
>
> El selector `MODELO_AGUA` de arriba les deja probar: TIP3P es con el que se parametrizó OL3; OPC es más moderno y reproduce mejor el agua pura. **No hay una respuesta correcta**, hay que evaluar cual es la mejor solución para cada sistema.
>

---
## Paso 4 - Solvatar: la forma de la caja importa

Metemos el complejo en agua explícita con **condiciones periódicas de frontera (PBC)**: la caja se repite infinitamente en las tres direcciones.

**Convención de imagen mínima.** Cada átomo interactúa sólo con la copia *más cercana* de cada otro átomo. Para que eso sea válido, la molécula **nunca** debe ver su propia imagen periódica.

**¿Y por qué una caja octaédrica en vez de un cubo?** Porque una biomolécula es más o menos globular, y un cubo desperdicia las esquinas: para garantizar la misma distancia mínima entre imágenes en todas las direcciones, un cubo necesita mucha más agua. Un **octaedro truncado** llena el espacio con ~77 % del volumen de un cubo equivalente, una **dodecaedro**, con ~71 %.

Ese ahorro es directamente tiempo de cómputo, porque el costo lo domina el solvente.

> ### Predicción
> Con 1.0 nm de margen: **¿cuántas moléculas de agua creen que caben?**
> ¿Decenas, cientos, miles? ¿Y cuánta agua nos ahorra el octaedro?

In [ ]:
#@title Solvatar y neutralizar
FORMA_CAJA = "octahedron"   #@param ["octahedron", "dodecahedron", "cube"]
PADDING_NM = 1            #@param {type:"number"}
SAL_MOLAR  = 0.15           #@param {type:"number"}
CATION     = "Na+"          #@param ["Na+", "K+", "Li+", "Rb+", "Cs+"]
ANION      = "Cl-"          #@param ["Cl-", "Br-", "F-", "I-"]

#@markdown - `FORMA_CAJA`: Forma de la caja
#@markdown - `PADDING_NM`: Distancia del borde de la caja a cualquier punto en la proteína o RNA, en nm
#@markdown - `SAL_MOLAR`: Concentración de sal **después** de neutralizar la carga
#@markdown - `CATION` Catión a elegir
#@markdown - `ANION`: Anion a eleger.

from collections import Counter
_t0 = time.time()

mod2 = Modeller(mod_h.topology, mod_h.positions)
mod2.addSolvent(forcefield, model=GEOM_AGUA, boxShape=FORMA_CAJA,
                padding=PADDING_NM*unit.nanometer,
                positiveIon=CATION, negativeIon=ANION,
                ionicStrength=SAL_MOLAR*unit.molar, neutralize=True)

top, pos = mod2.topology, mod2.positions
NOMBRES_ION = {"NA","CL","K","LI","RB","CS","BR","F","I"}
n_aguas = sum(1 for r in top.residues() if r.name in ("HOH","WAT"))
conteo_iones = Counter(r.name for r in top.residues() if r.name in NOMBRES_ION)

vec = np.array(top.getPeriodicBoxVectors().value_in_unit(unit.nanometer))
volumen = abs(np.linalg.det(vec))
anchos = np.array([volumen/np.linalg.norm(np.cross(vec[1], vec[2])),
                   volumen/np.linalg.norm(np.cross(vec[2], vec[0])),
                   volumen/np.linalg.norm(np.cross(vec[0], vec[1]))])

idx_soluto = [a.index for a in top.atoms()
              if a.residue.name not in ("HOH","WAT") and a.residue.name not in NOMBRES_ION]
xyz = np.array(pos.value_in_unit(unit.nanometer))[idx_soluto]
diametro = 2*np.linalg.norm(xyz - xyz.mean(axis=0), axis=1).max()
vol_cubo = (diametro + 2*PADDING_NM)**3

sis_tmp = forcefield.createSystem(top, nonbondedMethod=PME)
nb_tmp = [f for f in sis_tmp.getForces() if isinstance(f, mm.NonbondedForce)][0]
q_fin = sum(nb_tmp.getParticleParameters(i)[0].value_in_unit(unit.elementary_charge)
            for i in range(sis_tmp.getNumParticles()))

print("="*64)
print(f"  Átomos totales del sistema    : {top.getNumAtoms():8d}")
print(f"  Moléculas de agua             : {n_aguas:8d}")
print(f"  Iones                         : {dict(conteo_iones)}")
print(f"  CARGA TOTAL                   : {q_fin:+8.2f} e   <-- debe ser ~0")
print("-"*64)
print(f"  Forma de caja                 : {FORMA_CAJA}")
print(f"  Volumen                       : {volumen:8.1f} nm³")
print(f"  Un cubo equivalente sería     : {vol_cubo:8.1f} nm³ "
      f"({100*volumen/vol_cubo:.0f} % del cubo)")
print(f"  Agua ahorrada frente al cubo  : ~{100*(1-volumen/vol_cubo):.0f} %")
print("="*64)
print(f"\n(solvatación en {time.time()-_t0:.0f} s)")

with open("03_solvatado.pdb", "w") as f:
    PDBFile.writeFile(top, pos, f)

v = py3Dmol.view(width=800, height=480)
v.addModel(open("03_solvatado.pdb").read(), "pdb")

# Render water molecules as tiny spheres since they act as single atoms here
v.setStyle({"resn": ["HOH","WAT"]}, {"sphere": {"radius": 0.15, "color": C_AGUA, "opacity": 1}})

v.addStyle({"chain": "A"}, {"cartoon": {"color": C_PROT}})
v.addStyle({"chain": "B"}, {"cartoon": {"color": C_RNA}})

# Separate ions into positive and negative groups based on common naming conventions
pos_ions = [ion for ion in conteo_iones.keys() if any(p in ion.upper() for p in ['NA', 'K', 'MG', 'CA', 'ZN', '+'])]
neg_ions = [ion for ion in conteo_iones.keys() if any(n in ion.upper() for n in ['CL', 'CLA', 'BR', '-'])]

# Apply distinct colors (Blue for positive, Red for negative)
if pos_ions:
    v.addStyle({"resn": pos_ions}, {"sphere": {"radius": 0.8, "color": "#3b82f6"}}) # Blue
if neg_ions:
    v.addStyle({"resn": neg_ions}, {"sphere": {"radius": 0.8, "color": "#ef4444"}}) # Red

v.zoomTo(); v.show()
print("Azul = iones positivos | Rojo = iones negativos")

---
## Paso 5 - Minimización de energía

**La minimización no es dinámica molecular.** No hay temperatura, no hay velocidades, no hay tiempo. Es un algoritmo de optimización que baja por el gradiente de la superficie de energía potencial hasta el mínimo local más cercano.

¿Para qué sirve? Después de protonar y solvatar (incluso desde la estructura experimental) hay contactos incorrectos, hidrógenos recién colocados muy cerca, aguas extremadamente ordenadas. Si arrancáramos la dinámica en ese punto, esas fuerzas mandarían átomos a velocidades absurdas y la integración explotaría.

Minimizar remueve estos contactos antes de empezar.

> ### Predicción
> **¿La energía potencial va a subir o a bajar, y en qué orden de magnitud?**
> **¿Cuánto se va a mover la estructura: décimas de Å, o nanómetros?**

In [ ]:
#@title Construir el sistema y minimizar
CORTE_NM      = 1.0 #@param {type:"number"}
RESTRICCIONES = "HBonds"
AGUA_RIGIDA   = True
TEMPERATURA_K = 300        #@param {type:"number"}
FRICCION_PS   = 1.0
PASO_FS       = 2.0        #@param {type:"number"}
MAX_ITER_MIN  = 5000       #@param {type:"integer"}
PLATAFORMA    = "CUDA"

#@markdown - `CORTE_NM`: *Cutoff* para las interacciones no enlazadas, en nm
#@markdown - `TEMPERATURA_K`: Temperatura a la que se realizaraá la simulación, en K (se usa después de la minimización).
#@markdown - `PASO_FS`: Paso de integración, en fs.
#@markdown - `MAX_ITER_MIN`: Número máximos de ciclos de minimización.


_CONSTR = {"HBonds": HBonds, "AllBonds": AllBonds, "HAngles": HAngles, "None": None}

sistema = forcefield.createSystem(top, nonbondedMethod=PME,
                                  nonbondedCutoff=CORTE_NM*unit.nanometer,
                                  constraints=_CONSTR[RESTRICCIONES],
                                  rigidWater=AGUA_RIGIDA)
integrador = mm.LangevinMiddleIntegrator(TEMPERATURA_K*unit.kelvin,
                                         FRICCION_PS/unit.picosecond,
                                         PASO_FS*unit.femtoseconds)
sim = Simulation(top, sistema, integrador, mm.Platform.getPlatformByName(PLATAFORMA))
sim.context.setPositions(pos)

pos_ini = np.array(sim.context.getState(getPositions=True)
                   .getPositions().value_in_unit(unit.nanometer))
E_ini = sim.context.getState(getEnergy=True).getPotentialEnergy()

_t0 = time.time(); sim.minimizeEnergy(maxIterations=MAX_ITER_MIN); _dt = time.time()-_t0

E_fin = sim.context.getState(getEnergy=True).getPotentialEnergy()
pos_fin = np.array(sim.context.getState(getPositions=True)
                   .getPositions().value_in_unit(unit.nanometer))
desplaz = np.linalg.norm(pos_fin - pos_ini, axis=1)

print("="*60)
print(f"  Energía potencial ANTES   : {E_ini.value_in_unit(unit.kilojoule_per_mole):14.1f} kJ/mol")
print(f"  Energía potencial DESPUÉS : {E_fin.value_in_unit(unit.kilojoule_per_mole):14.1f} kJ/mol")
print(f"  Diferencia                : {(E_fin-E_ini).value_in_unit(unit.kilojoule_per_mole):14.1f} kJ/mol")
print("-"*60)
print(f"  Desplazamiento medio del soluto : {10*desplaz[idx_soluto].mean():.3f} Å")
print(f"  Desplazamiento máximo (todo)    : {10*desplaz.max():.3f} Å")
print(f"  (tiempo: {_dt:.0f} s)")
print("="*60)
print('''
Fíjense en la asimetría: la energía cae en decenas de miles de kJ/mol, pero los
átomos apenas se movieron. Toda esa energía estaba
concentrada en unos pocos contactos malos. La estructura no cambió: sólo se
quitaron las espinas.
''')
with open("04_minimizado.pdb", "w") as f:
    PDBFile.writeFile(top, sim.context.getState(getPositions=True).getPositions(), f)

In [ ]:
#@title VER  cuánto se movió el soluto al minimizar
#@markdown Gris = antes de minimizar · Color = después. Deberían verse casi encimados
import mdtraj as mdt
_m = mdt.load("04_minimizado.pdb")
_sel = _m.topology.select("not water and not resname NA CL K LI RB CS BR F I")
_m.atom_slice(_sel).save_pdb("04b_min_soluto.pdb")

v = py3Dmol.view(width=900, height=440)
v.addModel(open("02b_protonado.pdb").read(), "pdb")
v.setStyle({"model": 0}, {"cartoon": {"color": "#94a3b8"}})
v.addModel(open("04b_min_soluto.pdb").read(), "pdb")
v.setStyle({"model": 1}, {"cartoon": {"color": C_PROT}})
v.zoomTo(); v.show()
print("\n")

fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(10*desplaz[idx_soluto], bins=60, color=C_PROT, alpha=0.85)
ax.set_xlabel("desplazamiento al minimizar (Å)")
ax.set_ylabel("número de átomos"); ax.grid(alpha=0.25)
ax.set_title("La mayoría de los átomos se mueve poco")
plt.tight_layout(); plt.show()

---
## Paso 6 - Equilibración NVT (calentar)

Después de minimizar, todos los átomos tienen **velocidad cero**: el sistema está a 0 K.

> ### Pregunta
> **¿De dónde salen las velocidades? El sistema está congelado y nadie lo ha empujado.**

Se asignan al azar desde la distribución de Maxwell–Boltzmann a la temperatura objetivo. La condición inicial de una simulación de DM es aleatoria. Esta es la razón por la que **réplicas independientes con distintas semillas** son la forma correcta de estimar incertidumbre.

**Restricciones de posición:** durante el equilibrado restringimos los átomos pesados de los solutos con resortes, porque el agua todavía no se acomodó. Si dejamos todo libre desde el primer paso, la estructura se relaja hacia lo que el campo de fuerza prefiera, *mientras* el disolvente sigue desordenado, y perdemos la estructura experimental antes de empezar a medir.

In [ ]:
#@title Restricciones posicionales + equilibración NVT
PS_NVT           = 25     #@param {type:"number"}
K_RESTRICCION    = 500.0  #@param {type:"number"}
SEMILLA          = 1234   #@param {type:"integer"}
REPORTAR_CADA_PS = 0.5    #@param {type:"number"}

#@markdown - `PS_NVT`: Cuántos *ps* de calentamiento realizar
#@markdown - `K_RESTRICCION`: Fuerza de las restricciones sobre los átomos pesados (kJ/mol/A^2)
#@markdown - `SEMILLA`: Semilla para asignar velocidades iniciales (la misma semilla reproduce los mismos parámetros)
#@markdown - `REPORTAR_CADA_PS`: Intervalo de tiempo en el que se guardan los datos

import sys
pasos_nvt = int(PS_NVT*1000/PASO_FS)
rep_cada = max(1, int(REPORTAR_CADA_PS*1000/PASO_FS))
set_soluto = set(idx_soluto)

for i in reversed(range(sistema.getNumForces())):
    if isinstance(sistema.getForce(i), mm.CustomExternalForce):
        sistema.removeForce(i)

restr = mm.CustomExternalForce("k*periodicdistance(x, y, z, x0, y0, z0)^2")
restr.addGlobalParameter("k", K_RESTRICCION*unit.kilojoules_per_mole/unit.nanometer**2)
for p in ("x0","y0","z0"): restr.addPerParticleParameter(p)
ref = sim.context.getState(getPositions=True).getPositions()
n_restr = 0
for atom in top.atoms():
    if atom.index in set_soluto and atom.element != element.hydrogen:
        restr.addParticle(atom.index, ref[atom.index].value_in_unit(unit.nanometer))
        n_restr += 1
sistema.addForce(restr)
sim.context.reinitialize(preserveState=True)
print(f"Restricciones aplicadas a {n_restr} átomos pesados del soluto.\n")

integrador.setRandomNumberSeed(SEMILLA)
sim.context.setVelocitiesToTemperature(TEMPERATURA_K*unit.kelvin, SEMILLA)
print(f"Velocidades desde Maxwell-Boltzmann a {TEMPERATURA_K} K (semilla = {SEMILLA})")
print("Cambien la semilla y obtendrán otra trayectoria. Eso es una réplica.\n")

sim.reporters.clear()
sim.reporters.append(StateDataReporter("nvt.csv", rep_cada, step=True, time=True,
                                       potentialEnergy=True, kineticEnergy=True,
                                       totalEnergy=True, temperature=True, volume=True))
sim.reporters.append(StateDataReporter(sys.stdout, max(1, pasos_nvt//8), step=True,
                                       time=True, temperature=True,
                                       speed=True, separator="  |  "))
_t0 = time.time(); sim.step(pasos_nvt)
print(f"\nNVT terminado en {time.time()-_t0:.0f} s")
with open("05_nvt.pdb", "w") as f:
    PDBFile.writeFile(top, sim.context.getState(getPositions=True,
                      enforcePeriodicBox=True).getPositions(), f)

In [ ]:
#@title GRAFICAR - temperatura y energías durante el NVT
import pandas as pd
d = pd.read_csv("nvt.csv")
col = lambda k: d[[c for c in d.columns if k in c][0]]
t = col("Time")

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].plot(t, col("Temperature"), lw=1.1, color=C_PROT)
ax[0].axhline(TEMPERATURA_K, ls="--", c=C_RNA, lw=1.4, label=f"{TEMPERATURA_K} K objetivo")
ax[0].set_ylabel("Temperatura (K)"); ax[0].set_title("Temperatura"); ax[0].legend(fontsize=8)

ax[1].plot(t, col("Potential"), lw=1.1, color="#7c3aed")
ax[1].set_ylabel("E potencial (kJ/mol)"); ax[1].set_title("Energía potencial")

ax[2].plot(t, col("Kinetic"), lw=1.1, color="#dc2626", label="cinética")
ax[2].plot(t, col("Total"), lw=1.1, color="#0f766e", label="total")
ax[2].set_ylabel("Energía (kJ/mol)"); ax[2].set_title("Cinética y total"); ax[2].legend(fontsize=8)

for a in ax: a.set_xlabel("tiempo (ps)"); a.grid(alpha=0.25)
plt.suptitle("Equilibración NVT — # particulas, volumen y temperatura constante")
plt.tight_layout(); plt.show()

T = col("Temperature")
print(f"T promedio en la 2a mitad: {T[len(T)//2:].mean():.1f} ± {T[len(T)//2:].std():.1f} K")

La temperatura llega al objetivo en pocos *ps* y luego fluctúa alrededor. Esas fluctuaciones NO son un error numérico: en un sistema finito la temperatura
instantánea fluctúa como $\frac{1}{\sqrt{N}}$. Es termodinámica, no ruido.

La energía cinética sube (estamos calentando) mientras la potencial se acomoda. En NVT la energía total **NO** se conserva: el termostato permite un intercambio de energía a propósito.

---
## Paso 7 — Equilibración NPT

Hasta ahora el volumen estuvo fijo. Pero nosotros pusimos las aguas con una densidad que *inventamos*. No tiene por qué ser la correcta a 300 K y 1 bar de presión.

Activamos un **barostato**: el volumen de la caja puede cambiar, y el sistema encuentra por sí mismo su densidad de equilibrio.

In [ ]:
#@title Equilibración NPT con barostato
PS_NPT         = 50    #@param {type:"number"}
PRESION_BAR    = 1.0   #@param {type:"number"}
FREC_BAROSTATO = 25

#@markdown - `PS_NPT`: Duración de la equilibración NPT, en ps
#@markdown - `PRESION_BAR`: Presión objetivo para el barostato, en bar


pasos_npt = int(PS_NPT*1000/PASO_FS)

for i in reversed(range(sistema.getNumForces())):
    if isinstance(sistema.getForce(i), mm.MonteCarloBarostat):
        sistema.removeForce(i)

sistema.addForce(mm.MonteCarloBarostat(PRESION_BAR*unit.bar,
                                       TEMPERATURA_K*unit.kelvin, FREC_BAROSTATO))
sim.context.reinitialize(preserveState=True)

sim.reporters.clear()
sim.reporters.append(StateDataReporter("npt.csv", rep_cada, step=True, time=True,
                                       potentialEnergy=True, kineticEnergy=True,
                                       totalEnergy=True, temperature=True,
                                       volume=True, density=True))
sim.reporters.append(StateDataReporter(sys.stdout, max(1, pasos_npt//8), step=True,
                                       time=True, temperature=True, density=True,
                                       speed=True, separator="  |  "))
_t0 = time.time(); sim.step(pasos_npt)
print(f"\nNPT terminado en {time.time()-_t0:.0f} s")
with open("06_npt.pdb", "w") as f:
    PDBFile.writeFile(top, sim.context.getState(getPositions=True,
                      enforcePeriodicBox=True).getPositions(), f)

In [ ]:
#@title GRAFICAR - qué hace el barostato
d = pd.read_csv("npt.csv")
col = lambda k: d[[c for c in d.columns if k in c][0]]
t = col("Time")

fig, ax = plt.subplots(2, 2, figsize=(12, 6.6))
ax[0,0].plot(t, col("Density"), lw=1.1, color=C_PROT)
ax[0,0].axhline(0.997, ls="--", c=C_RNA, lw=1.4, label="agua pura, 300 K")
ax[0,0].set_ylabel("densidad (g/cm³)"); ax[0,0].set_title("Densidad"); ax[0,0].legend(fontsize=8)

ax[0,1].plot(t, col("Box Volume"), lw=1.1, color=C_RNA)
ax[0,1].set_ylabel("volumen (nm³)"); ax[0,1].set_title("Volumen de la caja")

ax[1,0].plot(t, col("Temperature"), lw=1.1, color="#dc2626")
ax[1,0].axhline(TEMPERATURA_K, ls="--", c=GRIS, lw=1.2)
ax[1,0].set_ylabel("Temperatura (K)"); ax[1,0].set_title("El termostato sigue activo")
ax[1,0].set_ylim(290, 310)

ax[1,1].plot(t, col("Potential"), lw=1.1, color="#7c3aed")
ax[1,1].set_ylabel("E potencial (kJ/mol)"); ax[1,1].set_title("Energía potencial")

for a in ax.ravel(): a.set_xlabel("tiempo (ps)"); a.grid(alpha=0.25)
plt.suptitle("Equilibración NPT — presión constante")
plt.tight_layout(); plt.show()

rho = col("Density"); m = len(rho)//2
print(f"Densidad promedio (2a mitad): {rho[m:].mean():.4f} ± {rho[m:].std():.4f} g/cm³")
print(f"Volumen promedio  (2a mitad): {col('Box Volume')[m:].mean():.1f} nm³")

---
## Paso 8 - Producción

Quitamos las restricciones posicionales y dejamos el sistema libre. Esta es la parte que vamos a analizar.

**Sobre el paso de integración.** El integrador tiene que resolver el movimiento más rápido del sistema: la vibración de estiramiento X–H, con período de unos 10 fs. Eso exigiría ~1 fs.

Pero pusimos `constraints=HBonds`, que fija la longitud de los enlaces a hidrógeno y elimina el problema de los movimientos más rápidos.


> ### Pregunta
> A 2 fs por paso: ¿cuántos pasos son 200 ps? ¿Y 100 ns?
>
> **¿Qué les dice esto sobre lo que están a punto de obtener?**

In [ ]:
#@title Producción - corrida corta
PS_PRODUCCION   = 200   #@param {type:"number"}
GUARDAR_CADA_PS = 10    #@param {type:"number"}

#@markdown - `PS_PRODUCCION`: Duración de la simulación, en ps
#@markdown - `GUARDAR_CADA_PS`: Intervalo para guardar los datos.

#@markdown **DETALLE**: 200 ps es un tiempo muy corto, el sistema sigue equilibrandose. Esto solo es un ejercicio para que alcance a terminar durante el taller.
#@markdown Para análizar una simulación hoy, les voy a pasar una precalculada de 100 ns del mismo sistema.

pasos_prod = int(PS_PRODUCCION*1000/PASO_FS)
guardar_cada = max(1, int(GUARDAR_CADA_PS*1000/PASO_FS))

for i in reversed(range(sistema.getNumForces())):
    if isinstance(sistema.getForce(i), mm.CustomExternalForce):
        sistema.removeForce(i)
sim.context.reinitialize(preserveState=True)
print("Restricciones posicionales eliminadas. El soluto está libre.\n")
print(f"{pasos_prod:,} pasos x {PASO_FS} fs = {PS_PRODUCCION} ps")
print(f"Se guardarán {pasos_prod//guardar_cada} cuadros (uno cada {GUARDAR_CADA_PS} ps)")
print(f"Para comparar: 100 ns = {int(100000*1000/PASO_FS):,} pasos, "
      f"{int(100000/PS_PRODUCCION)}x más de lo que haremos aquí.\n")

sim.reporters.clear()
sim.reporters.append(DCDReporter("produccion_corta.dcd", guardar_cada))
sim.reporters.append(StateDataReporter("produccion.csv", guardar_cada, step=True,
                                       time=True, potentialEnergy=True, temperature=True,
                                       density=True, volume=True))
sim.reporters.append(StateDataReporter(sys.stdout, max(1, pasos_prod//10), step=True,
                                       time=True, temperature=True,
                                       speed=True, separator="  |  "))
_t0 = time.time(); sim.step(pasos_prod); _dt = time.time()-_t0

print(f"\nProducción terminada en {_dt/60:.1f} min")
vel = (PS_PRODUCCION/1000)/(_dt/86400)
print(f"Rendimiento: {vel:.1f} ns/día")
print(f"A este ritmo, 100 ns tomarían {100/vel:.1f} días en este Colab.")
print("Ésa es exactamente la razón por la que la trayectoria larga viene pre-calculada.")

with open("07_topologia_final.pdb", "w") as f:
    PDBFile.writeFile(top, sim.context.getState(getPositions=True,
                      enforcePeriodicBox=True).getPositions(), f)

---
## Paso 9 — Centrar y reimaginar la trayectoria

Antes de mirar nada hay un paso que casi nadie explica y que produce las imágenes más
confusas de toda la dinámica molecular.

En una caja periódica **las moléculas no se quedan dentro de la caja**. Cuando un átomo
sale por un lado, el programa lo reintroduce por el opuesto. Si uno visualiza el archivo
tal cual, ve moléculas partidas en pedazos, o que "saltan" de un lado a otro.

Nada de eso es físico: es un artefacto de cómo se guardan las coordenadas.

**Reimaginar** (*imaging*) significa reconstruir cada molécula entera y volver a centrar
el soluto. **Alinear** significa quitar la traslación y la rotación globales, que no nos
interesan. Sólo después de eso las imágenes significan algo.

In [ ]:
#@title Centrar, reimaginar y alinear
ALINEAR_SOBRE = "name CA" # @param ["name CA","name P","name CA or name P",""] {"allow-input":true}

import subprocess, textwrap, os

SEL_STRIP = ":HOH,WAT,NA,CL,K,LI,RB,CS,BR"

guion = textwrap.dedent(f"""\
    parm 07_topologia_final.pdb
    trajin produccion_corta.dcd

    #autoimage anchor :31-131
    center :1-131
    image familiar
    rms first {'@CA' if ALINEAR_SOBRE == 'name CA' else '@P' if ALINEAR_SOBRE == 'name P' else '@CA,P'}

    # version CON disolvente, para ver la caja
    outtraj 08_octaedro_con_agua.pdb pdb
    outtraj 08_trayectoria_agua.xtc

    # fuera disolvente y contraiones
    strip {SEL_STRIP}

    trajout 08_trayectoria_centrada.xtc
    trajout 08_topologia_centrada.pdb pdb onlyframes 1

    go
    quit
    """)

with open("autoimage.in", "w") as f:
    f.write(guion)

r = subprocess.run(["cpptraj", "-i", "autoimage.in"], capture_output=True, text=True)
print(r.stdout[-2500:])
if r.returncode != 0:
    print(r.stderr[-2500:])
    raise RuntimeError("cpptraj falló — revisen la salida de arriba.")

traj = mdt.load("08_trayectoria_centrada.xtc", top="08_topologia_centrada.pdb")
print(f"\nProcesada: {traj.n_frames} cuadros, {traj.n_atoms} átomos, "
      f"alineada sobre '{ALINEAR_SOBRE}'")

In [ ]:
#@title VER - la caja octaédrica
v = py3Dmol.view(width=800, height=520)
v.addModel(open("08_octaedro_con_agua.pdb").read(), "pdb", {"keepH": True})
v.setStyle({"resn": ["HOH", "WAT"]}, {"sphere": {"radius": 0.12, "color": C_AGUA}})
v.addStyle({"chain": "A"}, {"cartoon": {"color": C_PROT}})
v.addStyle({"chain": "B"}, {"cartoon": {"color": C_RNA}})
v.zoomTo(); v.show()
print("Ahora sí se ve el octaedro truncado. Giren la vista para apreciar las caras.")

In [ ]:
#@title VER - su simulación, animada **SIN AGUA**
traj = mdt.load("/content/08_trayectoria_centrada.xtc", top="/content/08_topologia_centrada.pdb")
# 1. Downsample the trajectory (e.g., every 10th frame) to avoid freezing the browser
traj_subset = traj

# 2. Save and load via temporary file
temp_file = "temp_traj_view.pdb"
traj_subset.save_pdb(temp_file)

with open(temp_file, "r") as f:
    pdb_string = f.read()

os.remove(temp_file)

# 3. Initialize viewer and load frames
view = py3Dmol.view(width=800, height=600)
view.addModelsAsFrames(pdb_string, "pdb")

# 4. Apply your specific styles across ALL frames ('model': -1)
view.setStyle({'model': -1, 'chain': 'B'}, {'cartoon': {'color': C_PROT}})
view.setStyle({'model': -1, 'chain': 'A'}, {'cartoon': {'color': C_RNA}})
view.addStyle({'model': -1, 'chain': 'A'}, {'stick': {'radius': 0.12, 'color': C_RNA}})

# 5. Animate and render
view.animate({'loop': 'forward', 'reps': 0, 'interval': 100})
view.zoomTo()
view.show()
print(f"Ésta es su simulación. Cada cuadro son {GUARDAR_CADA_PS} ps de movimiento real.")
print('''
Miren con atención cuál de las dos moléculas se mueve más. Esa observación cualitativa, hecha a ojo, es exactamente la que vamos a cuantificar en el Ejercicio 2.
''')

In [ ]:
#@title VER - su simulación, animada **CON AGUA**
traj = mdt.load("/content/08_trayectoria_agua.xtc", top="/content/08_octaedro_con_agua.pdb")
# 1. Downsample the trajectory (e.g., every 10th frame) to avoid freezing the browser
traj_subset = traj

# 2. Save and load via temporary file
temp_file = "temp_traj_view.pdb"
traj_subset.save_pdb(temp_file)

with open(temp_file, "r") as f:
    pdb_string = f.read()

os.remove(temp_file)

# 3. Initialize viewer and load frames
view = py3Dmol.view(width=800, height=600)
view.addModelsAsFrames(pdb_string, "pdb", {'keepH': True})

# 4. Apply your specific styles across ALL frames ('model': -1)
view.setStyle({'model': -1, 'chain': 'B'}, {'cartoon': {'color': C_PROT}})
view.setStyle({'model': -1, 'chain': 'A'}, {'cartoon': {'color': C_RNA}})
view.addStyle({'model': -1, 'chain': 'A'}, {'stick': {'radius': 0.12, 'color': C_RNA}})

# 5. Animate and render
view.animate({'loop': 'forward', 'reps': 0, 'interval': 100})
view.zoomTo()
view.show()
print(f"Ésta es su simulación. Cada cuadro son {GUARDAR_CADA_PS} ps de movimiento real.")
print('''
Miren con atención cuál de las dos moléculas se mueve más. Esa observación cualitativa, hecha a ojo, es exactamente la que vamos a cuantificar en el Ejercicio 2.
''')

---
## ¿Qué obtuvimos realmente?

Una sola medición antes de terminar: el RMSD respecto a la estructura inicial, por
separado para la proteína y para el RNA.

In [ ]:
#@title RMSD de la corrida corta
rmsd_prot = 10*mdt.rmsd(traj, traj, 0, atom_indices=traj.topology.select("name CA"))
rmsd_rna  = 10*mdt.rmsd(traj, traj, 0, atom_indices=traj.topology.select("name P"))
t = np.arange(traj.n_frames)*GUARDAR_CADA_PS

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(t, rmsd_prot, "o-", lw=1.4, ms=4, color=C_PROT, label="proteína (Cα)")
ax.plot(t, rmsd_rna,  "o-", lw=1.4, ms=4, color=C_RNA,  label="RNA (P)")
ax.set_xlabel("tiempo (ps)"); ax.set_ylabel("RMSD (Å)")
ax.set_title(f"RMSD durante {PS_PRODUCCION} ps — ¿ven una meseta?")
ax.legend(); ax.grid(alpha=0.25); plt.tight_layout(); plt.show()

print(f"RMSD final proteína : {rmsd_prot[-1]:.2f} Å")
print(f"RMSD final RNA      : {rmsd_rna[-1]:.2f} Å")
print('''
Las dos curvas todavía están subiendo. No hay meseta.

Si alguien les entregara esta gráfica diciendo "el complejo es estable",
¿le creerían? ¿Qué podrían afirmar honestamente con estos datos, y qué no?
''')

---
## Guardar el trabajo

En el Cuaderno 2 vamos a comparar su corrida contra la trayectoria de 100 ns
**y contra las corridas de las demás parejas.**

In [ ]:
#@title Empaquetar y descargar resultados
NOMBRE = f"resultados_modelo{MODELO_NMR}"
!tar -czf {NOMBRE}.tar.gz 08_trayectoria_centrada.xtc 08_topologia_centrada.pdb 08_octaedro_con_agua.pdb 08_trayectoria_agua.xtc\
        produccion_corta.dcd 07_topologia_final.pdb nvt.csv npt.csv produccion.csv 2>/dev/null
!ls -lh {NOMBRE}.tar.gz

try:
    from google.colab import files
    files.download(f"{NOMBRE}.tar.gz")
except Exception as e:
    print("Descarga manual desde el panel de archivos:", e)

---

## Para llevarse

1. **La estructura de partida es una decisión, no un dato.** La RMN depositó un ensamble
   de modelos igualmente válidos. Ustedes eligieron uno, y esa elección se propaga a todo.
2. **Elegir bien la estructura evita más error que cualquier arreglo posterior.**
3. **Los hidrógenos no vienen del experimento.** Son casi la mitad del sistema y los
   pusimos nosotros, a un pH que elegimos.
4. **La forma de la caja es una decisión de costo.** El octaedro truncado da la misma
   distancia mínima entre imágenes con ~23 % menos agua que un cubo.
5. La minimización **no es** dinámica: baja mucha energía moviendo muy poco.
6. Las velocidades iniciales son **aleatorias**. Por eso se corren réplicas.
7. "Equilibrado" no es propiedad del sistema: es propiedad **de un observable**.
   La densidad convergió en 30 ps; el RMSD no había convergido en 200 ps.
8. El paso de 2 fs es consecuencia de haber congelado los enlaces X–H.
9. **Una trayectoria sin reimaginar produce imágenes falsas.** Moléculas partidas y
   saltos por la pantalla son artefactos de la periodicidad, no física.
10. **200 ps no alcanzan para casi nada.** No es un defecto del taller: es el problema
    central del campo.

**Cuaderno 2:** la misma simulación, 500 veces más larga — y las corridas de todas las
parejas juntas.